# HW2 — Thompson Sampling

**Course:** ID6002W Online and Reinforcement Learning (IIT Madras WSAI web-enabled M.Tech)

**Deadline:** Wednesday, 17 June 2026, 11:59 PM.

**Marks:** 4 (Parts A–D, 1 mark each).


## About this homework

HW2 has two parts, in separate notebooks. **This notebook covers Thompson Sampling (TS) from Lectures 5 and 6** — the randomized counterpart to UCB. It is broken into four short graded parts:

- **Part A** — does TS work? Cumulative regret on a clean instance.
- **Part B** — why does it work? Watch the Beta posteriors concentrate over time.
- **Part C** — tune the prior. What happens when the prior is wrong?
- **Part D** — tune the decay. A discount factor lets old data fade in a non-stationary world.

In every part the pattern is the same:

1. **Implement** the core vanilla Thompson Sampling logic once in the `ts` function.
2. **Run** the pre-filled experiment and plotting cells.
3. **Comment** on what you see in a short write-up cell.

Your code is needed to produce the plots, but **marks are awarded for the written observations** — your interpretation of what the plots show and why. The marks for each part are stated in that part's heading. The plotting and experiment scaffolding is provided.

**Prerequisite:** Task 1 of this homework, where you implemented UCB. Part A reuses your UCB function once, as a reference curve against TS — copy your finished `ucb` into the cell indicated below before running Part A. The rest of the notebook does not use UCB.

Part D uses a provided `discounted_ts` function. You are not being asked to implement discounted TS; the goal there is to interpret the effect of forgetting old observations.


## Provided codebase (from HW1)

The next four cells provide imports, `BernoulliBandit`, plotting helpers, and `run_experiment`. These are identical to HW1 — included here so this homework is self-contained. Run them once, then move on to the task.

A further environment, `DriftingBernoulliBandit`, is introduced later in Part D, right before the experiments that use it.


In [ ]:
# === Setup: imports and plotting style ===

import numpy as np                  # numerical arrays + random number generators
import matplotlib.pyplot as plt     # plotting
from functools import partial       # used later to "bind" hyperparameters (e.g. ε) into algorithms

%matplotlib inline

# A few plot defaults so figures look consistent across the notebook.
plt.rcParams["figure.dpi"] = 100    # higher dpi = sharper figures
plt.rcParams["axes.grid"] = True    # show grid by default
plt.rcParams["grid.alpha"] = 0.3    # faded grid lines so they don't dominate

# Fixed starting seed for reproducibility (see warmup for explanation).
BASE_SEED = 20260516

In [ ]:
# === Provided: BernoulliBandit environment ===

class BernoulliBandit:
    """
    K-arm Bernoulli bandit.

    Parameters
    ----------
    means : sequence of floats in [0, 1]
        True mean reward of each arm.
    rng : np.random.Generator
        Source of randomness owned by this environment instance.

    Attributes
    ----------
    K : int                       Number of arms.
    means : np.ndarray (K,)       True means.
    best_mean : float             max of `means` (used for regret).
    best_arm : int                argmax of `means`.
    pull_counts : np.ndarray (K,) Number of times each arm has been pulled.
    """

    def __init__(self, means, rng):
        # Store true means as a numpy array of floats.
        self.means = np.asarray(means, dtype=float)

        # Sanity check: Bernoulli means must be valid probabilities.
        if np.any((self.means < 0) | (self.means > 1)):
            raise ValueError("Bernoulli means must lie in [0, 1].")

        self.K = len(self.means)               # number of arms
        self.rng = rng                          # this env's own random generator

        # Pre-compute best arm and its mean — used to compute regret.
        self.best_mean = float(self.means.max())
        self.best_arm = int(np.argmax(self.means))

        # Bookkeeping: how often each arm has been pulled so far.
        # (Algorithms may also track this internally; this is for sanity-checking.)
        self.pull_counts = np.zeros(self.K, dtype=int)

    def pull(self, arm):
        """Pull arm `arm`. Returns a 0/1 reward sampled Bernoulli(means[arm])."""
        self.pull_counts[arm] += 1
        # rng.random() draws uniform in [0, 1); reward = 1 with probability means[arm].
        return int(self.rng.random() < self.means[arm])

    def instantaneous_regret(self, arm):
        """Expected regret incurred by pulling arm `arm`: best_mean − means[arm].

        Note this is the *expected* regret, not the realised one. We use expected
        regret throughout because it's smoother and the standard quantity to plot.
        """
        return self.best_mean - self.means[arm]

In [ ]:
# === Provided: plotting helpers ===

def plot_cumulative_regret(regrets, label, ax):
    """Plot cumulative regret for a single run.

    Parameters
    ----------
    regrets : (T,) instantaneous regret per round (not cumulative — we sum here).
    """
    ax.plot(np.cumsum(regrets), label=label)   # cumsum turns per-round regret into cumulative
    ax.set_xlabel("Round t")
    ax.set_ylabel("Cumulative regret")
    ax.legend()


def plot_averaged_regret(regret_matrix, label, ax, with_band=True):
    """Plot mean cumulative regret across seeds, with optional ±1 std band.

    Parameters
    ----------
    regret_matrix : (n_seeds, T) instantaneous regret per round per seed.
    """
    # Each row is one run; cumsum along time gives (n_seeds, T) cumulative regrets.
    cum = np.cumsum(regret_matrix, axis=1)

    # Aggregate across seeds: mean curve plus standard-deviation band.
    mean = cum.mean(axis=0)
    std = cum.std(axis=0)

    t = np.arange(1, cum.shape[1] + 1)
    line, = ax.plot(t, mean, label=label)
    if with_band:
        # Shaded band shows variability across seeds — wider band = noisier algorithm.
        ax.fill_between(t, mean - std, mean + std, alpha=0.2, color=line.get_color())
    ax.set_xlabel("Round t")
    ax.set_ylabel("Cumulative regret  (mean ± 1 std)")
    ax.legend()


def plot_arm_pulls(arm_sequence, ax, K=None):
    """Strip plot: which arm was pulled at which round.

    Useful for visualising the exploration pattern of an algorithm.

    Parameters
    ----------
    arm_sequence : (T,) integer arm indices.
    """
    arm_sequence = np.asarray(arm_sequence)
    t = np.arange(len(arm_sequence))
    ax.scatter(t, arm_sequence, s=6, alpha=0.5)   # small, semi-transparent dots
    ax.set_xlabel("Round t")
    ax.set_ylabel("Arm pulled")
    if K is None:
        K = int(arm_sequence.max()) + 1
    ax.set_yticks(np.arange(K))   # one tick per arm

In [ ]:
# === Provided: run_experiment helper ===

def run_experiment(algo_fn, env_factory, T, n_seeds, base_seed=BASE_SEED):
    """Run `algo_fn` for `n_seeds` independent seeds.

    Parameters
    ----------
    algo_fn : callable
        Signature `algo_fn(env, T) -> (arm_sequence, regret_sequence)`.
        Bind any extra params (e.g. ε for ε-greedy) with `functools.partial`
        before passing.
    env_factory : callable
        Signature `env_factory(rng) -> BernoulliBandit`. Builds a fresh env per
        seed — so each seed sees an independent stream of rewards.
    T : int                Horizon.
    n_seeds : int          Number of independent runs to average over.
    base_seed : int        Seeds used are base_seed + 0, base_seed + 1, ...

    Returns
    -------
    regret_matrix : (n_seeds, T) float array of instantaneous regrets.
    arm_matrix    : (n_seeds, T) int   array of arms pulled.
    """
    # Pre-allocate output matrices: one row per seed, one column per round.
    regret_matrix = np.zeros((n_seeds, T))
    arm_matrix = np.zeros((n_seeds, T), dtype=int)

    for s in range(n_seeds):
        # Each run gets its own RNG, derived from base_seed + s.
        # This makes the whole experiment reproducible while keeping runs independent.
        rng = np.random.default_rng(base_seed + s)
        env = env_factory(rng)

        # Run the algorithm. It returns the arms it pulled and the per-round regret.
        arms, regrets = algo_fn(env, T)

        regret_matrix[s] = regrets
        arm_matrix[s] = arms

    return regret_matrix, arm_matrix

---


In [ ]:
# === Provided in Task 1: UCB ===
#
# Part A compares vanilla TS against your UCB function from Task 1. Copy your
# finished `ucb` implementation here, with the same signature you used in
# Task 1, before running Part A. The rest of this notebook does not use UCB.

def ucb(env, T, c=2.0, mode="fixed_horizon", sigma=1.0, T_assumed=None,
        return_trace=False):
    # TODO: paste the body of your UCB implementation from Task 1 here.
    raise NotImplementedError(
        "Paste your UCB implementation from Task 1 in place of this stub."
    )


In [ ]:
# === Task 2: Thompson Sampling implementation ===

def ts(env, T, prior_alpha=1.0, prior_beta=1.0, return_trace=False):
    """Vanilla Thompson Sampling for a Bernoulli bandit.

    For each arm i, maintain a Beta(alphas_i, betas_i) posterior over its mean.
    Each round, sample one plausible mean from each posterior and pull the arm
    whose sample is largest. Then update only that arm's posterior.

        each round:
            theta_tilde_i ~ Beta(alphas_i, betas_i)           for each arm i
            arm = argmax_i theta_tilde_i
            observe reward r in {0, 1}
            alphas[arm] += r
            betas[arm]  += (1 - r)

    Parameters
    ----------
    env          : bandit with .K, .pull(arm), .instantaneous_regret(arm), .rng
    T            : int     horizon (number of rounds to play)
    prior_alpha  : float or length-K array   initial alpha(s)
    prior_beta   : float or length-K array   initial beta(s)
    return_trace : bool    if True, also return per-round (alpha, beta) arrays
                            (used by the Part B posterior-evolution plot).

    Returns
    -------
    arm_sequence    : (T,) int     arm pulled at each round
    regret_sequence : (T,) float   instantaneous (expected) regret per round
    trace           : dict         returned ONLY if return_trace=True. Keys "alpha"
                                    and "beta", each a (T, K) array of the posterior
                                    parameters at the end of each round.
    """
    K = env.K

    arm_sequence = np.zeros(T, dtype=int)
    regret_sequence = np.zeros(T)

    # Beta(alphas_i, betas_i) belief over each arm's mean. Initialised to the prior.
    # Multiplication by np.ones(K) allows prior_alpha/prior_beta to be either scalars
    # or length-K arrays.
    alphas = np.asarray(prior_alpha, dtype=float) * np.ones(K)
    betas = np.asarray(prior_beta, dtype=float) * np.ones(K)

    # Provided: storage for the Part B plot. Do not edit.
    if return_trace:
        alpha_hist = np.zeros((T, K))
        beta_hist = np.zeros((T, K))

    for t_idx in range(T):
        # TODO: implement one round of vanilla Thompson Sampling.
        #   1. Sample: draw one plausible mean from each arm's current posterior.
        #      env.rng.beta is vectorised: env.rng.beta(alphas, betas) returns a
        #      length-K array of samples in a single call.
        #   2. Greedy under the sampled world: set `arm` to the index of the
        #      largest sample (np.argmax).
        #   3. Observe and update:
        #         reward = env.pull(arm)
        #         add reward to alphas[arm] and (1 - reward) to betas[arm].
        # By the end of this block, `arm` must be defined.
        arm = ...   # TODO
        raise NotImplementedError("Implement the Thompson Sampling round above.")

        # Provided: record bookkeeping. Do not edit.
        arm_sequence[t_idx] = arm
        regret_sequence[t_idx] = env.instantaneous_regret(arm)

        # Provided: record the posterior for the Part B plot. Do not edit.
        if return_trace:
            alpha_hist[t_idx] = alphas
            beta_hist[t_idx] = betas

    if return_trace:
        return arm_sequence, regret_sequence, {"alpha": alpha_hist, "beta": beta_hist}
    return arm_sequence, regret_sequence


### Part A — does TS work? *(1 mark)*

Vanilla TS on a clean 3-arm Bernoulli instance with means $\mu = (0.2, 0.55, 0.7)$ — one clear winner, one near-miss, one clearly bad. We add a tuned UCB curve from Task 1 to the same plot as a reference point. Both algorithms are expected to have logarithmic-type regret on this instance, so seeing them in the same ballpark is a useful sanity check that TS is behaving reasonably. Part B will reuse this exact instance to look at the posteriors that produced this regret curve.


In [ ]:
# === Task 2, Part A: cumulative regret of vanilla TS, with UCB as reference ===

T = 20000
n_seeds = 50

def env_factory(rng):
    """3-arm Bernoulli: one clear winner, one near-miss, one clearly bad."""
    return BernoulliBandit([0.2, 0.55, 0.7], rng)

# Vanilla TS: default prior Beta(1, 1) on every arm.
ts_regret, ts_arms = run_experiment(ts, env_factory, T, n_seeds)

# Tuned UCB reference. We use c=0.25 here, lower than the more conservative
# textbook-style values, because the goal is only to anchor TS against another
# sensible algorithm on this particular instance.
ucb_regret, _ = run_experiment(partial(ucb, c=0.25), env_factory, T, n_seeds)

fig, ax = plt.subplots(figsize=(9, 5))
plot_averaged_regret(ucb_regret, "UCB (c=0.25, tuned reference)", ax)
plot_averaged_regret(ts_regret, "vanilla TS", ax)
ax.set_title(f"UCB vs vanilla TS on 3-arm Bernoulli  μ=[0.2, 0.55, 0.7]  (T={T}, {n_seeds} seeds)")
plt.show()

# Average pull counts for TS, plus the lecture's scaling check T_i(T) * Delta_i^2.
# Theory predicts this product is roughly arm-independent across suboptimal arms.
means_true = np.array([0.2, 0.55, 0.7])
mu_star = means_true.max()
pull_counts = np.array([(ts_arms == k).sum(axis=1) for k in range(3)]).mean(axis=1)
gaps = mu_star - means_true                                  # Delta_i; 0 for best arm

print(f"{'arm':<8}{'true mean':>12}{'avg pulls (TS)':>18}{'T_i(T)·Δ_i²':>16}")
for k, mu in enumerate(means_true):
    scaled = pull_counts[k] * gaps[k] ** 2
    scaled_str = f"{scaled:>16.2f}" if gaps[k] > 0 else f"{'—':>16}"
    print(f"arm {k:<6}{mu:>10.2f}{pull_counts[k]:>18.0f}{scaled_str}")


**Write-up (TODO) — 1 mark.**

Both curves should be sublinear and visibly slowing — consistent with logarithmic-type regret for both algorithms on this instance. Lecture 4 went further than the rate: it showed that for UCB, the expected number of pulls of a suboptimal arm $i$ satisfies $T_i(T) \le 8 \log T / \Delta_i^2 + 1$, where $\Delta_i = \mu^\star - \mu_i$ is the gap. The same inverse-square-in-the-gap scaling is also the right intuition for TS — even though TS does not compute any explicit confidence interval. Check this against your pull counts: compute $T_i(T) \cdot \Delta_i^2$ for the two suboptimal arms and report the values. They should be in the same ballpark, even though one arm was pulled much more often than the other. Then explain in two or three lines *why* the gap appears squared and not linearly. (Hint: the standard deviation of a Beta posterior after $n$ pulls of an arm shrinks roughly as $1/\sqrt{n}$. Roughly how many pulls do you need before this width drops below the gap $\Delta_i$ — the precision required to reliably tell arm $i$ apart from the best arm? Solve for $n$.)

*(TODO: write your observations below.)*


### Part B — watch the posteriors concentrate *(1 mark)*

Part A confirmed from the outside that TS works: the regret curve is in the same ballpark as tuned UCB. Part B looks *inside*. The `ts` function we wrote supports `return_trace=True`, which records the Beta parameters $(\alpha_i, \beta_i)$ for every arm at every round. We re-run TS once on the same 3-arm instance from Part A, with one seed, and watch the posteriors evolve over time.

A quick refresher on Beta$(\alpha, \beta)$ as a belief over an arm's success probability $p_i$:

- Posterior mean $= \alpha / (\alpha + \beta)$ — our current best guess of $p_i$.
- Beta$(1, 1)$ is uniform on $[0, 1]$ — total uncertainty, the prior we start from.
- A success increments $\alpha$, pulling probability mass right; a failure increments $\beta$, pulling it left.
- The width depends on $\alpha + \beta$: the more evidence we have, the narrower the posterior — the standard deviation shrinks roughly as $1/\sqrt{\alpha + \beta}$, which for a long run is roughly $1/\sqrt{N_i(t)}$.

The ridgeline plot below visualises the three Beta posteriors at several time snapshots between $t = 0$ and $t = T$. Each ridge is height-normalised independently, so read the **width** of the ridge as the uncertainty.


In [ ]:
# === Task 2, Part B: run TS once with posterior tracing on ===
# Same 3-arm instance as Part A, single seed, so we can look inside one run.

from scipy.stats import beta as beta_dist   # for evaluating Beta pdfs

T = 20000
SEED = BASE_SEED

rng = np.random.default_rng(SEED)
env = BernoulliBandit([0.2, 0.55, 0.7], rng)
arms_run, regrets_run, trace = ts(env, T, return_trace=True)

# Posterior parameter histories: shape (T, K).
alpha_hist = trace["alpha"]
beta_hist = trace["beta"]

# Cumulative per-arm pull counts: shape (T, K).
counts_per_arm = np.cumsum(
    np.stack([arms_run == k for k in range(env.K)], axis=1), axis=0
)

# Shared x-grid for evaluating Beta pdfs (avoid 0 and 1 for stability).
xs = np.linspace(1e-3, 1 - 1e-3, 400)

def posterior_pdfs(t_idx):
    """Return (K, len(xs)) array of Beta pdf values at round t_idx (0-indexed)."""
    out = np.zeros((env.K, len(xs)))
    for k in range(env.K):
        out[k] = beta_dist.pdf(xs, alpha_hist[t_idx, k], beta_hist[t_idx, k])
    return out

def pulls_at(t_idx):
    """Cumulative pulls of each arm up to and including round t_idx."""
    return [int(counts_per_arm[t_idx, k]) for k in range(env.K)]

# Summary print: final pulls and final Beta parameters per arm.
print(f"Final pull counts (t = T): {pulls_at(T - 1)}")
print(f"{'arm':<6}{'μ':>8}{'α(T)':>12}{'β(T)':>12}{'post. mean':>14}")
for k in range(env.K):
    a, b = alpha_hist[-1, k], beta_hist[-1, k]
    print(f"arm {k:<3}{env.means[k]:>8.2f}{a:>12.1f}{b:>12.1f}{a/(a+b):>14.4f}")


In [ ]:
# === Task 2, Part B: ridgeline plot of posteriors over time ===
# Each snapshot's height is normalised independently so stacks do not occlude;
# the story is in the *width* (narrower = more concentrated) and the *N* annotations.

ridge_times = [25, 100, 500, 2000, 5000, 10000, 20000]
colors = ["#1f77b4", "#2ca02c", "#d62728"]
OFFSET = 1.0                                # vertical gap between stacked snapshots
SCALE_CAP = 0.85                            # each ridge's peak height in plot units

fig, axes = plt.subplots(1, 3, figsize=(15, 6.5), sharey=True)
for k, ax in enumerate(axes):
    for j, t in enumerate(ridge_times):
        t_idx = t - 1
        a, b = alpha_hist[t_idx, k], beta_hist[t_idx, k]
        pdf = beta_dist.pdf(xs, a, b)
        pdf_scaled = SCALE_CAP * pdf / max(pdf.max(), 1e-9)
        y_base = j * OFFSET
        ax.fill_between(xs, y_base, y_base + pdf_scaled,
                        color=colors[k], alpha=0.45, lw=0)
        ax.plot(xs, y_base + pdf_scaled, color=colors[k], lw=1)
        # Per-snapshot N annotation on the right edge.
        n_here = int(counts_per_arm[t_idx, k])
        ax.text(1.01, y_base + SCALE_CAP * 0.5, f"N={n_here}",
                va="center", ha="left", fontsize=8, color="gray")

    ax.axvline(env.means[k], color="black", ls=":", lw=1, alpha=0.7)
    pulls_T = pulls_at(T - 1)[k]
    ax.set_title(f"arm {k}   μ = {env.means[k]}   ·   N(T) = {pulls_T}")
    ax.set_xlim(0, 1)
    ax.set_xlabel("possible mean μ")
    ax.set_yticks([j * OFFSET + SCALE_CAP * 0.5 for j in range(len(ridge_times))])
    ax.set_yticklabels([f"t = {t}" for t in ridge_times])

axes[0].set_ylabel("time snapshot")
fig.suptitle(
    f"Beta posteriors over time, per arm — single seed, T = {T}\n"
    f"(each snapshot's height is normalised independently; read the *width* "
    f"of each ridge as the posterior uncertainty)"
)
plt.tight_layout()
plt.show()


**Write-up (TODO) — 1 mark.**

**Question 1 (concentration over time).** Describe how each arm's posterior changes from early time to $t = T$. The standard deviation of a Beta posterior after $N$ pulls of an arm shrinks roughly as $1/\sqrt{N}$ (the same scaling you used in Part A). Plug in the actual final pull counts $N_i(T)$ printed above, predict the rough width of each arm's posterior at $t = T$, and check this against what you see in the plot.

**Question 2 (asymmetric narrowing).** Arm 0 ($\mu = 0.2$) is the *worst* arm, yet its posterior stays the widest of the three. Arm 1 ($\mu = 0.55$) is only slightly worse than arm 2 ($\mu = 0.7$), yet its posterior eventually concentrates clearly. Why does TS bother narrowing arm 1's posterior but not arm 0's? (Hint: TS only ever pulls an arm if a sample from its posterior wins the argmax against samples from the other arms. At $t = 5000$, can a sample from arm 1's posterior plausibly beat a typical sample from arm 2's posterior? Can a sample from arm 0's?)

*(TODO: write your observations below.)*


### Part C — prior sensitivity *(1 mark)*

So far we have used the uninformative prior Beta(1, 1) on every arm — the uniform distribution on $[0, 1]$, which says "we know nothing about any arm's mean." In a deployed system we often do know something. A bandit chosen to replace an existing recommender might start with strong beliefs about each arm based on historical data. Part C asks: what does the choice of prior do to TS, and what happens if the prior is wrong?

A Beta$(\alpha, \beta)$ prior is equivalent to having already observed $\alpha - 1$ phantom successes and $\beta - 1$ phantom failures. So Beta$(7, 3)$ acts as if we have seen $6$ successes in $8$ phantom trials, a moderately strong belief that the underlying mean is near $0.7$. Beta$(3, 7)$ is the same strength on the other side.

We test three prior settings on a clean 2-arm Bernoulli instance with means $\mu = [0.3, 0.7]$:

1. **Uninformative:** Beta$(1, 1)$ on both arms. The baseline used in Parts A and B.
2. **Informative and correct:** Beta$(3, 7)$ on arm 0, Beta$(7, 3)$ on arm 1. Each arm's prior is centred at its own true mean, with moderate strength.
3. **Informative and wrong:** Beta$(7, 3)$ on arm 0, Beta$(3, 7)$ on arm 1. The two priors are swapped, so we begin believing arm 0 is the good arm and arm 1 is the bad one. This is the scenario worth understanding: how much damage does a wrong prior do, and does TS recover?

The `ts` function accepts `prior_alpha` and `prior_beta` either as scalars (same prior on all arms) or as length-K arrays (per-arm priors). We use the array form in this part.


In [ ]:
# === Task 2, Part C: prior sensitivity on a 2-arm Bernoulli ===

T = 5000
n_seeds = 50

def env_factory_2arm(rng):
    return BernoulliBandit([0.3, 0.7], rng)

# Each setting is (label, prior_alpha vector, prior_beta vector).
prior_settings = [
    ("uninformative Beta(1,1)",        np.array([1.0, 1.0]), np.array([1.0, 1.0])),
    ("correct: Beta(3,7), Beta(7,3)",  np.array([3.0, 7.0]), np.array([7.0, 3.0])),
    ("wrong:   Beta(7,3), Beta(3,7)",  np.array([7.0, 3.0]), np.array([3.0, 7.0])),
]

regret_by_prior = {}
for label, pa, pb in prior_settings:
    inst_regret, _ = run_experiment(
        partial(ts, prior_alpha=pa, prior_beta=pb),
        env_factory_2arm, T, n_seeds,
    )
    regret_by_prior[label] = inst_regret

# Plot 1: cumulative regret under the three priors.
fig, ax = plt.subplots(figsize=(10, 5.5))
for label, _, _ in prior_settings:
    plot_averaged_regret(regret_by_prior[label], label, ax)
ax.set_title(
    f"TS prior sensitivity on 2-arm Bernoulli μ=[0.3, 0.7]  "
    f"(T={T}, {n_seeds} seeds)"
)
plt.show()


In [ ]:
# === Task 2, Part C: posterior snapshot at t = 200, one panel per prior setting ===
# A single seed, so we can look at posteriors actually produced by one run.

T_SNAP = 200
SEED = BASE_SEED
xs = np.linspace(1e-3, 1 - 1e-3, 400)
arm_colors = ["#1f77b4", "#d62728"]
true_means = [0.3, 0.7]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, (label, pa, pb) in zip(axes, prior_settings):
    rng = np.random.default_rng(SEED)
    env = env_factory_2arm(rng)
    arms_run, _, trace = ts(env, T_SNAP, prior_alpha=pa, prior_beta=pb,
                            return_trace=True)
    alpha_h = trace["alpha"]
    beta_h = trace["beta"]
    for k in range(env.K):
        a_T, b_T = alpha_h[-1, k], beta_h[-1, k]
        a_0, b_0 = pa[k], pb[k]
        n_pulls = int((arms_run == k).sum())
        ax.plot(xs, beta_dist.pdf(xs, a_T, b_T), color=arm_colors[k], lw=2,
                label=f"arm {k}: posterior at t={T_SNAP}, N={n_pulls}")
        ax.plot(xs, beta_dist.pdf(xs, a_0, b_0), color=arm_colors[k], lw=1,
                ls="--", alpha=0.5,
                label=f"arm {k}: prior Beta({a_0:.0f},{b_0:.0f})")
        ax.axvline(true_means[k], color=arm_colors[k], ls=":", lw=1, alpha=0.6)
    ax.set_xlim(0, 1)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("possible mean μ")
    ax.legend(loc="upper center", fontsize=8)
axes[0].set_ylabel("posterior density")
fig.suptitle(f"Beta posteriors at t = {T_SNAP}, one seed per prior setting")
plt.tight_layout()
plt.show()


**Write-up (TODO) — 1 mark.**

A Beta$(\alpha, \beta)$ prior acts like having already observed $\alpha - 1$ phantom successes and $\beta - 1$ phantom failures. Using this, explain (i) why TS with the correct informative prior accumulates almost no early regret, and (ii) why TS with the wrong informative prior still recovers eventually.

*(TODO: write your observations below.)*


### Part D — discounted TS for a non-stationary environment *(1 mark)*

Parts A, B, C all assumed that the arm means stay fixed for the whole run. Real-world deployments are usually not like this. In Lecture 5 we discussed DoorDash's use of TS to choose responsive Dashers, and one of the reasons for using a bandit method was non-stationarity: user demand and Dasher behaviour can shift over time. A method that keeps exploring and can update its beliefs as new data arrives is suited to this setting, but only if it is set up to forget old data when needed.

**The problem with the basic TS update.** In the vanilla update used in Parts A, B, C, the pseudo-counts $\alpha_i$ and $\beta_i$ accumulate every observation ever made. After many rounds the posterior is concentrated, with combined weight $\alpha_i + \beta_i \approx N_i(t) + 2$, while a single new observation enters with weight one. In a stationary environment this is the right behaviour: every observation contains information about the same fixed $\mu_i$. In a non-stationary environment it can be harmful. If $\mu_i$ has changed, old observations no longer reflect the same quantity as new ones, but the posterior still gives them their original weight, so it moves only slowly toward the new value.

**One standard fix: discounted TS.** We now use a separate provided function, `discounted_ts`. The idea is to discount old evidence but keep the original prior alive. Before each new observation, we replace

$$
\alpha_i \leftarrow \alpha_0 + \gamma(\alpha_i - \alpha_0), \qquad
\beta_i \leftarrow \beta_0 + \gamma(\beta_i - \beta_0),
$$

where $(\alpha_0, \beta_0)$ is the prior. Thus the accumulated evidence $\alpha_i - \alpha_0$ and $\beta_i - \beta_0$ decays geometrically, while an ignored arm drifts back toward the neutral prior rather than toward a pathological Beta$(0,0)$-like object.

**Effective memory length.** A useful summary number is the *effective memory length*, defined as the number of rounds $N$ at which the weight $\gamma^N$ has dropped to $1/e \approx 0.368$. Setting $\gamma^N = 1/e$ gives $N \approx 1/(1-\gamma)$ for $\gamma$ close to $1$. So $\gamma = 0.99$ gives roughly $100$ rounds of effective memory, $\gamma = 0.999$ gives roughly $1000$, and $\gamma = 0.95$ gives roughly $20$.

**A new environment for this part.** We define `DriftingBernoulliBandit` below. It has the same interface as `BernoulliBandit` and so works with `run_experiment` without any change. Internally it maintains a round counter and switches its arm means at a chosen round $T_{\mathrm{switch}}$.


In [ ]:
# === Task 2, Part D: discounted TS and an abrupt-switch environment ===

def discounted_ts(env, T, gamma=0.99, prior_alpha=1.0, prior_beta=1.0,
                  return_trace=False):
    """Discounted Thompson Sampling for a Bernoulli bandit.

    This is provided for Part D. It discounts accumulated evidence relative to
    the prior:

        alpha <- prior_alpha + gamma * (alpha - prior_alpha)
        beta  <- prior_beta  + gamma * (beta  - prior_beta)

    So old observations fade, but an ignored arm returns to the prior rather
    than toward Beta(0, 0).
    """
    K = env.K
    if not (0 < gamma <= 1):
        raise ValueError("gamma must lie in (0, 1].")

    prior_alpha_vec = np.asarray(prior_alpha, dtype=float) * np.ones(K)
    prior_beta_vec = np.asarray(prior_beta, dtype=float) * np.ones(K)

    alphas = prior_alpha_vec.copy()
    betas = prior_beta_vec.copy()

    arm_sequence = np.zeros(T, dtype=int)
    regret_sequence = np.zeros(T)

    if return_trace:
        alpha_hist = np.zeros((T, K))
        beta_hist = np.zeros((T, K))

    for t_idx in range(T):
        # Discount accumulated evidence, while keeping the prior alive.
        alphas = prior_alpha_vec + gamma * (alphas - prior_alpha_vec)
        betas = prior_beta_vec + gamma * (betas - prior_beta_vec)

        theta_tilde = env.rng.beta(alphas, betas)
        arm = int(np.argmax(theta_tilde))

        reward = env.pull(arm)
        alphas[arm] += reward
        betas[arm] += (1 - reward)

        arm_sequence[t_idx] = arm
        regret_sequence[t_idx] = env.instantaneous_regret(arm)

        if return_trace:
            alpha_hist[t_idx] = alphas
            beta_hist[t_idx] = betas

    if return_trace:
        return arm_sequence, regret_sequence, {"alpha": alpha_hist, "beta": beta_hist}
    return arm_sequence, regret_sequence


class DriftingBernoulliBandit:
    """Bernoulli bandit whose arm means switch abruptly at a fixed round.

    Same interface as BernoulliBandit, so run_experiment treats it identically.
    The current phase is determined by an internal round counter, which is
    incremented on every pull().
    """

    def __init__(self, means_phase1, means_phase2, t_switch, rng):
        assert len(means_phase1) == len(means_phase2)
        self._means_phase1 = np.asarray(means_phase1, dtype=float)
        self._means_phase2 = np.asarray(means_phase2, dtype=float)
        self._t_switch = int(t_switch)
        self.K = len(means_phase1)
        self.rng = rng
        self._t = 0                                 # internal round counter

    @property
    def means(self):
        """Current arm means, depending on the phase."""
        return self._means_phase1 if self._t < self._t_switch else self._means_phase2

    def pull(self, arm):
        reward = float(self.rng.random() < self.means[arm])
        self._t += 1                                # advance after observing
        return reward

    def instantaneous_regret(self, arm):
        """Regret in the current phase: mu_star_now - mu_arm_now."""
        m = self.means
        return float(m.max() - m[arm])


In [ ]:
# === Task 2, Part D: discount-factor sweep on an abrupt-switch environment ===

T = 10000
T_SWITCH = 5000
n_seeds = 50

def drift_env_factory(rng):
    """2-arm bandit; means swap at t = T_SWITCH."""
    return DriftingBernoulliBandit(
        means_phase1=[0.7, 0.3],
        means_phase2=[0.3, 0.7],
        t_switch=T_SWITCH,
        rng=rng,
    )

gammas = [1.0, 0.999, 0.99, 0.95]
regret_by_gamma = {}
arms_by_gamma = {}
for g in gammas:
    inst_regret, arms = run_experiment(
        partial(discounted_ts, gamma=g), drift_env_factory, T, n_seeds
    )
    regret_by_gamma[g] = inst_regret
    arms_by_gamma[g] = arms

# Plot 1: cumulative regret vs t, one line per gamma.
fig, ax = plt.subplots(figsize=(10, 5.5))
for g in gammas:
    eff_mem = "∞" if g == 1.0 else f"≈{int(round(1 / (1 - g)))}"
    plot_averaged_regret(
        regret_by_gamma[g], f"γ = {g}  (eff. memory {eff_mem})", ax
    )
ax.axvline(T_SWITCH, color="black", ls="--", lw=1, alpha=0.6)
ax.text(T_SWITCH, ax.get_ylim()[1] * 0.02, "  switch", fontsize=9, color="black")
ax.set_title(
    f"Discounted TS on an abrupt-switch environment "
    f"(μ flips at t={T_SWITCH}; T={T}, {n_seeds} seeds)"
)
plt.show()


In [ ]:
# === Task 2, Part D: rolling action fraction after the switch ===
# This plot shows behaviour, not just regret: how quickly does the algorithm
# stop selecting arm 0, which was optimal before the switch but bad afterward?

def rolling_action_fraction(arm_matrix, arm, window):
    """Mean rolling fraction of choosing `arm`, averaged over seeds."""
    indicators = (arm_matrix == arm).astype(float)
    rolled = []
    for row in indicators:
        c = np.cumsum(np.insert(row, 0, 0.0))
        rolled.append((c[window:] - c[:-window]) / window)
    return np.vstack(rolled).mean(axis=0)

WINDOW = 200
t_axis = np.arange(WINDOW, T + 1)
true_old_best_indicator = np.where(t_axis <= T_SWITCH, 1.0, 0.0)

fig, ax = plt.subplots(figsize=(10, 5.5))
for g in [1.0, 0.99]:
    frac_arm0 = rolling_action_fraction(arms_by_gamma[g], arm=0, window=WINDOW)
    label = "γ = 1.0 (vanilla TS)" if g == 1.0 else "γ = 0.99 (discounted TS)"
    ax.plot(t_axis, frac_arm0, lw=2, label=label)

ax.plot(t_axis, true_old_best_indicator, color="black", ls="--", lw=1.5,
        label="arm 0 is optimal? (1 before switch, 0 after)")
ax.axvline(T_SWITCH, color="black", ls=":", lw=1, alpha=0.6)
ax.set_xlim(WINDOW, T)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("Round t")
ax.set_ylabel(f"Rolling fraction of pulls of arm 0 (window = {WINDOW})")
ax.set_title("Does TS abandon the old best arm after the environment changes?")
ax.legend(loc="best")
plt.show()


**Write-up (TODO) — 1 mark.**

The discount factor $\gamma$ controls how quickly old observations are forgotten: an observation from $N$ rounds ago contributes roughly $\gamma^N$ to the current posterior evidence. Using the regret plot and the rolling-action plot, explain (i) why $\gamma = 0.99$ recovers from the switch at $t = 5000$ much faster than $\gamma = 1.0$, and (ii) why $\gamma = 0.95$ is not strictly better than $\gamma = 0.99$, even though it has a shorter effective memory.

*(TODO: write your observations below.)*
